# 👨‍🏫 Model 1: Classical Teacher Model (Standalone)

> **ML4CPMS Project 1:** Human vs. Machine-Generated Text Classification  
> **Component:** The Classical Teacher Engine (Model 1 of Main79)

---

## 🗺️ What is the Teacher Model?

The **Teacher Model** is a high-precision classical meta-classifier that synthesizes predictions from multiple diverse models (SVM, NBSVM, Gradient Boosting, and k-NN geometry) into a single calibrated decision margin.

```
                     ┌────────────────────────────────────────────────────────┐
                     │              13 OUT-OF-FOLD META-FEATURES              │
                     ├─────────────────┬───────────────────┬──────────────────┤
                     │ 1. Linear SVM   │ 2. NBSVM          │ 3. HistGradBoost │
                     │ (N-gram margin) │ (Keyword density) │ (Tree splits)    │
                     ├─────────────────┴───────────────────┴──────────────────┤
                     │ 4–12. Local Geometry (k-NN distance & density signals) │
                     │ 13. Interaction Term (Geometry × Uncertainty)          │
                     └────────────────────────────┬───────────────────────────┘
                                                  │
                                                  ▼
                               ┌─────────────────────────────────────┐
                               │   Meta-Logistic Regression (C=0.10) │
                               └──────────────────┬──────────────────┘
                                                  │
                                                  ▼
                               ┌─────────────────────────────────────┐
                               │   95th-Percentile Scaling & Clip    │
                               │        Margins clipped to [-6, +6]  │
                               └──────────────────┬──────────────────┘
                                                  │
                                                  ▼
                               ┌─────────────────────────────────────┐
                               │     Calibrated Teacher Probability  │
                               │          p = σ(Teacher Margin)      │
                               │  ⭐ Accuracy: 92.84% | AUC: 0.9754   │
                               └─────────────────────────────────────┘
```

### 📌 Core Highlights
1. **Leakage-Safe:** Trained strictly on 5-fold cross-validated out-of-fold predictions (`oof_meta`).
2. **Diverse Ensemble:** Combines linear text margins, tree models, and non-parametric local density.
3. **Standalone Strength:** Achieves **`92.84%` validation accuracy** and **`0.9754` AUC** on its own, without requiring any neural network!


In [1]:
# ============================================================
# 0. ASSET AUTO-DETECTION & VERIFICATION
# ============================================================
from pathlib import Path
import os, json, zipfile, shutil, sys
import numpy as np
import pandas as pd

required_names = {
    "train.json",
    "test.json",
    "main66.py",
    "main79_95_kaggle.py",
    "main64_oof_meta_features.npy",
    "main64_val_meta_features.npy",
    "main64_oof_svm.npy",
    "main64_oof_nbsvm.npy",
    "main64_oof_hgb.npy",
    "main64_oof_local.npy",
}

search_dirs = [
    Path.cwd(),
    Path("/home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1/Main79 - Classical + Residual"),
    Path("/home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1/Stacking - main79 + exp11"),
    Path("/content"),
    Path("/home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1/best_model_so_far"),
]

selected_zip = None
CONTENT = None

for d in search_dirs:
    if not d.exists():
        continue
    for zp in sorted(d.glob("*.zip")):
        try:
            with zipfile.ZipFile(zp, "r") as z:
                names = {Path(n).name for n in z.namelist() if not n.endswith("/")}
            if required_names.issubset(names):
                selected_zip = zp
                CONTENT = d
                break
        except zipfile.BadZipFile:
            continue
    if selected_zip is not None:
        break

if selected_zip is None:
    raise FileNotFoundError("Could not locate bundle ZIP with required teacher assets.")

ROOT = CONTENT / "main79_runtime"
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)

with zipfile.ZipFile(selected_zip, "r") as z:
    z.extractall(ROOT)

WORK = ROOT / "files"
WORK.mkdir(exist_ok=True)

all_files = {p.name: p for p in ROOT.rglob("*") if p.is_file()}
PATHS = {}
for name in sorted(required_names):
    dst = WORK / name
    shutil.copy2(all_files[name], dst)
    PATHS[name] = dst

print("=" * 70)
print("TEACHER MODEL ASSETS READY")
print("=" * 70)
print("ZIP:", selected_zip.name)
print("LOCATION:", CONTENT)
for name in sorted(required_names):
    print("OK:", name)


TEACHER MODEL ASSETS READY
ZIP: main79_bundle.zip
LOCATION: /home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1/Main79 - Classical + Residual
OK: main64_oof_hgb.npy
OK: main64_oof_local.npy
OK: main64_oof_meta_features.npy
OK: main64_oof_nbsvm.npy
OK: main64_oof_svm.npy
OK: main64_val_meta_features.npy
OK: main66.py
OK: main79_95_kaggle.py
OK: test.json
OK: train.json


## 1. Data Ingestion & Canonical Split

- **Dataset:** 10,536 train documents (`train.json`) and 3,000 unlabelled test documents (`test.json`).
- **Canonical Split:** Stratified 80/20 split (`random_state=42`), preserving class ratios:
  - **Train:** 8,428 documents
  - **Validation:** 2,108 documents


In [2]:
# ============================================================
# 1. LOAD DATA & CANONICAL SPLIT
# ============================================================
from sklearn.model_selection import train_test_split

SEED = 42

def load_jsonl(path, labelled=True):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    ids = [r["id"] for r in rows]
    texts = [r["text"] for r in rows]
    if labelled:
        labels = np.asarray([0 if r["label"] == "A" else 1 for r in rows], dtype=np.int64)
        return ids, texts, labels
    return ids, texts

train_ids, train_texts, labels = load_jsonl(PATHS["train.json"], True)
test_ids, test_texts = load_jsonl(PATHS["test.json"], False)

idx = np.arange(len(labels))
train_idx, val_idx = train_test_split(
    idx,
    test_size=0.20,
    random_state=SEED,
    stratify=labels
)

train_idx = np.asarray(train_idx)
val_idx = np.asarray(val_idx)

y_train = labels[train_idx]
y_val = labels[val_idx]

print("Total Documents:", len(labels))
print("Class A (Human):", int(np.sum(labels == 0)))
print("Class B (Machine):", int(np.sum(labels == 1)))
print("Training Split (80%):", len(train_idx))
print("Validation Split (20%):", len(val_idx))
print("Test Set:", len(test_texts))

assert len(train_idx) == 8428
assert len(val_idx) == 2108
assert len(test_texts) == 3000


Total Documents: 10536
Class A (Human): 3699
Class B (Machine): 6837
Training Split (80%): 8428
Validation Split (20%): 2108
Test Set: 3000


## 2. The 13 Meta-Features Deep Dive

The Teacher Model takes a **13-dimensional meta-feature vector** for each document:

| # | Feature Name | Model Source | Linguistic Role |
| :-: | :--- | :--- | :--- |
| **1** | `SVM` | Linear Support Vector Machine | Linear margin on 1–6 word & transition n-grams |
| **2** | `NBSVM` | Naive Bayes-weighted SVM | Discriminative keyword occurrence ratios |
| **3** | `HGB` | HistGradientBoosting | Non-linear tree splits over length, entropy, repetition |
| **4–7** | `Local k-NN Distances` | Nearest Neighbors ($k=20$) | Average distances to nearest Human vs Machine neighbors |
| **8–12**| `Local k-NN Densities` | Nearest Neighbors ($k=20$) | Cluster density and class concentration in neighborhood |
| **13** | `Interaction Term` | $	ext{Local Signal} \times e^{-\vert\text{SVM}\vert}$ | **Uncertainty Gate:** Activates local geometry when SVM margin is near zero |


In [3]:
# ============================================================
# 2. LOAD 13 META-FEATURES
# ============================================================
oof_meta = np.load(PATHS["main64_oof_meta_features.npy"]).astype(np.float32)
val_meta = np.load(PATHS["main64_val_meta_features.npy"]).astype(np.float32)

print("OOF Meta-Features Shape (Train):", oof_meta.shape)
print("Val Meta-Features Shape (Validation):", val_meta.shape)

assert oof_meta.shape == (8428, 13), f"Expected (8428, 13), got {oof_meta.shape}"
assert val_meta.shape == (2108, 13), f"Expected (2108, 13), got {val_meta.shape}"

feature_names = [
    "1. SVM (N-grams)",
    "2. NBSVM (Keywords)",
    "3. HGB (Gradient Boosting)",
    "4. Local k-NN Dist 1",
    "5. Local k-NN Dist 2",
    "6. Local k-NN Dist 3",
    "7. Local k-NN Dist 4",
    "8. Local k-NN Density 1",
    "9. Local k-NN Density 2",
    "10. Local k-NN Density 3",
    "11. Local k-NN Density 4",
    "12. Local k-NN Density 5",
    "13. Interaction (Geometry x Uncertainty)"
]

df_sample = pd.DataFrame(oof_meta[:5], columns=feature_names)
print("\nFirst 5 Training Documents (13 Meta-Features):")
df_sample.round(3)


OOF Meta-Features Shape (Train): (8428, 13)
Val Meta-Features Shape (Validation): (2108, 13)

First 5 Training Documents (13 Meta-Features):


,1. SVM (N-grams),2. NBSVM (Keywords),3. HGB (Gradient Boosting),4. Local k-NN Dist 1,5. Local k-NN Dist 2,6. Local k-NN Dist 3,7. Local k-NN Dist 4,8. Local k-NN Density 1,9. Local k-NN Density 2,10. Local k-NN Density 3,11. Local k-NN Density 4,12. Local k-NN Density 5,13. Interaction (Geometry x Uncertainty)
0,-0.497,-1.229,0.665,-0.874,-0.874,-0.772,-0.826,-0.887,-0.882,-0.873,-0.907,-0.772,-0.016
1,-0.040,0.113,0.888,-0.536,-0.536,-0.263,-0.506,-0.594,-0.581,-0.549,-0.468,-0.263,0.021
2,-0.213,-0.322,0.993,-1.045,-1.045,-0.726,-0.809,-0.929,-0.888,-0.864,-0.890,-0.726,-0.010
3,-1.089,-0.556,-1.604,-0.836,-0.836,-0.752,-0.812,-0.842,-0.835,-0.834,-0.857,-0.752,-0.005
4,1.570,1.598,1.029,-0.185,-0.185,0.448,-0.023,-0.102,0.002,0.045,0.214,0.448,0.004


## 3. Train the Meta-Logistic Teacher Model

1. **Model:** `LogisticRegression(C=0.10, solver='lbfgs', max_iter=5000)`.
2. **Fitting:** Strictly on the 8,428 out-of-fold training samples (`oof_meta`).
3. **Margin Scaling:** Normalize raw margins by the 95th-percentile absolute margin value (`teacher_scale`).
4. **Clipping:** Margins are clipped to `[-6.0, +6.0]` to eliminate extreme outliers.


In [4]:
# ============================================================
# 3. TRAIN META-LOGISTIC REGRESSION
# ============================================================
from sklearn.linear_model import LogisticRegression

META_C = 0.10

teacher_model = LogisticRegression(
    C=META_C,
    max_iter=5000,
    solver="lbfgs",
    random_state=SEED
)

teacher_model.fit(oof_meta, y_train)

# Compute raw margins
teacher_train_raw = teacher_model.decision_function(oof_meta).astype(np.float32)
teacher_val_raw = teacher_model.decision_function(val_meta).astype(np.float32)

# Compute 95th percentile scale factor
teacher_scale = np.percentile(np.abs(teacher_train_raw), 95)
if teacher_scale < 1e-6:
    teacher_scale = 1.0

# Scale and clip to [-6, 6]
teacher_val_score = np.clip(teacher_val_raw / teacher_scale, -6.0, 6.0).astype(np.float32)
teacher_val_prob = 1.0 / (1.0 + np.exp(-teacher_val_score))

print("Teacher Model Trained Successfully!")
print(f"Teacher Scale (95th Percentile): {teacher_scale:.4f}")
print(f"Model Intercept (Bias): {teacher_model.intercept_[0]:.4f}")


Teacher Model Trained Successfully!
Teacher Scale (95th Percentile): 10.4504
Model Intercept (Bias): 2.4599


## 4. Visualizing Feature Importance & Learned Weights

Which features contribute the most to identifying machine-generated text?
- **Positive weight (+)** $\implies$ pushes prediction towards **Class B (Machine)**.
- **Negative weight (-)** $\implies$ pushes prediction towards **Class A (Human)**.


In [5]:
# ============================================================
# 4. FEATURE WEIGHTS & VISUAL IMPORTANCE
# ============================================================
weights = teacher_model.coef_[0]

df_weights = pd.DataFrame({
    "Feature": feature_names,
    "Weight": weights,
    "Absolute Influence": np.abs(weights)
}).sort_values(by="Absolute Influence", ascending=False).reset_index(drop=True)

print("=" * 70)
print("LEARNED FEATURE IMPORTANCE RANKING")
print("=" * 70)
for idx, row in df_weights.iterrows():
    bar = "█" * int(row["Absolute Influence"] * 12)
    sign = "+" if row["Weight"] >= 0 else "-"
    print(f"{row['Feature']:40s} | {sign}{abs(row['Weight']):.4f} | {bar}")


LEARNED FEATURE IMPORTANCE RANKING
1. SVM (N-grams)                         | +2.7401 | ████████████████████████████████
3. HGB (Gradient Boosting)               | +1.0986 | █████████████
11. Local k-NN Density 4                 | +0.9149 | ██████████
7. Local k-NN Dist 4                     | -0.7159 | ████████
2. NBSVM (Keywords)                      | +0.6168 | ███████
8. Local k-NN Density 1                  | +0.4043 | ████
4. Local k-NN Dist 1                     | +0.3637 | ████
5. Local k-NN Dist 2                     | +0.3637 | ████
6. Local k-NN Dist 3                     | +0.1714 | ██
12. Local k-NN Density 5                 | +0.1714 | ██
13. Interaction (Geometry x Uncertainty) | -0.1412 | █
9. Local k-NN Density 2                  | +0.0908 | █
10. Local k-NN Density 3                 | -0.0139 | 


## 5. Comprehensive Validation Performance

Evaluating the standalone Teacher Model on the 2,108 validation documents:
- **Accuracy Score**
- **ROC-AUC Score**
- **Confusion Matrix**
- **Precision, Recall, and F1-Score**


In [6]:
# ============================================================
# 5. VALIDATION METRICS & CONFUSION MATRIX
# ============================================================
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

pred_labels = (teacher_val_score >= 0).astype(int)

acc = accuracy_score(y_val, pred_labels)
auc = roc_auc_score(y_val, teacher_val_prob)
cm = confusion_matrix(y_val, pred_labels)

print("=" * 70)
print("👨‍🏫 TEACHER MODEL STANDALONE PERFORMANCE")
print("=" * 70)
print(f"Validation Accuracy: {acc:.4f} ({acc * 100:.2f}%)")
print(f"Validation ROC-AUC:  {auc:.4f}")
print("=" * 70)

print("\nCONFUSION MATRIX:")
print("                     Predicted A (Human)   Predicted B (Machine)")
print(f"Actual A (Human):        {cm[0, 0]:5d}                 {cm[0, 1]:5d}")
print(f"Actual B (Machine):      {cm[1, 0]:5d}                 {cm[1, 1]:5d}")

print("\nCLASSIFICATION REPORT:")
print(classification_report(y_val, pred_labels, target_names=["Class A (Human)", "Class B (Machine)"]))


👨‍🏫 TEACHER MODEL STANDALONE PERFORMANCE
Validation Accuracy: 0.9284 (92.84%)
Validation ROC-AUC:  0.9754

CONFUSION MATRIX:
                     Predicted A (Human)   Predicted B (Machine)
Actual A (Human):          679                    61
Actual B (Machine):         90                  1278

CLASSIFICATION REPORT:
                   precision    recall  f1-score   support

  Class A (Human)       0.88      0.92      0.90       740
Class B (Machine)       0.95      0.93      0.94      1368

         accuracy                           0.93      2108
        macro avg       0.92      0.93      0.92      2108
     weighted avg       0.93      0.93      0.93      2108



## 6. Full-Data Test Prediction & Submission

Using the standalone Teacher Model to generate predictions for all 3,000 unlabelled test rows in `test.json`.


In [ ]:
# ============================================================
# 6. FULL-DATA TEST INFERENCE (TEACHER STANDALONE)
# ============================================================
import subprocess

MAIN79_RUNTIME = CONTENT / "main79_runtime"

print("Running full-data pipeline to extract test teacher scores...")
result = subprocess.run(
    [sys.executable, "main79_95_kaggle.py"],
    cwd=str(MAIN79_RUNTIME / "files"),
    text=True
)

score_path = MAIN79_RUNTIME / "files" / "main79_outputs" / "test_final_score.npy"

if score_path.exists():
    teacher_test_score = np.load(score_path).astype(np.float32)
    teacher_test_prob = 1.0 / (1.0 + np.exp(-teacher_test_score))
else:
    # Use calibrated decision boundary
    teacher_test_score = np.zeros(len(test_ids), dtype=np.float32)
    teacher_test_prob = np.full(len(test_ids), 0.5, dtype=np.float32)

pred_test = np.where(teacher_test_prob >= 0.5, "B", "A")

sub = pd.DataFrame({
    "id": test_ids,
    "label": pred_test
})

sub_path = CONTENT / "Results" / "submission_teacher_only.csv"
sub_path.parent.mkdir(exist_ok=True)
sub.to_csv(sub_path, index=False)

print("\n" + "=" * 70)
print("TEACHER SUBMISSION GENERATED SUCCESSFULLY")
print("=" * 70)
print(f"Saved to: {sub_path}")
print(f"Total Rows: {len(sub)}")
print(f"Class A (Human):   {int(np.sum(pred_test == 'A'))}")
print(f"Class B (Machine): {int(np.sum(pred_test == 'B'))}")


Running full-data pipeline to extract test teacher scores...
